# Entregable 3 — Sistema multiagente (Grupo 2)
Navegante · Crítico-Corrector · Árbitro, todos con **Qwen2.5-1.5B-Instruct**. Ejecutar en orden con **GPU T4** (*Entorno de ejecución → Cambiar tipo*).
Los resultados y trazas se guardan en Drive y la corrida es **reanudable**: si Colab se corta, vuelvan a ejecutar desde la celda 5.

## 1. Código y carpeta de resultados

In [ ]:
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/GenIA_Entregable3"
except ImportError:
    OUT_DIR = "resultados_entregable3"
os.makedirs(OUT_DIR, exist_ok=True)

REPO_URL = "https://github.com/Cata-X/GenIA.git"
SUBCARPETA = "GenIA_D3"          # <- ajustar si el codigo quedo en la raiz del repo
if not os.path.exists("src"):
    if not os.path.exists("GenIA"):
        !git clone -q {REPO_URL}
    base = os.path.join("GenIA", SUBCARPETA)
    os.chdir(base if os.path.exists(base) else "GenIA")
sys.path.insert(0, os.getcwd())
print("Directorio:", os.getcwd(), "| resultados en:", OUT_DIR)

In [ ]:
!pip -q install -r requirements.txt

## 2. Pruebas (sin modelo, segundos)
Comprueban que los textos que genera el codigo son identicos a los ejemplos de los prompts.

In [ ]:
!python -m pytest -q tests

## 3. Cargar el modelo (el mismo del D2, fijado por hash)

In [ ]:
from src.llm import QwenLLM, MODEL_NAME, MODEL_REVISION
llm = QwenLLM()
MODELO = f"{MODEL_NAME}@{MODEL_REVISION}"
print("Modelo cargado:", MODELO)

## 4. Prueba de humo (3 laberintos, con mensajes en vivo)
Mirar: ¿el Navegante copia la coordenada de la línea? ¿el Crítico acierta (ver contadores)? ¿cuánto tarda un laberinto?
Si algún prompt hay que cambiarlo, **crear `_v2`** y subir `version_prompts` en `Config`; no editar `_v1`.

In [ ]:
from src.maze import bfs, dibujar
from src.evaluate import cargar_dataset
from src.orchestrator import resolver, CONFIGS, Tracer

datos = cargar_dataset(None, 20)
for x in datos[:3]:
    lab, a, b = x["laberinto"], tuple(x["inicio"]), tuple(x["meta"])
    print(f"\n===== Laberinto {x['id']} | inicio {a} meta {b} | optimo {len(bfs(lab,a,b))-1} pasos =====")
    r = resolver(llm, lab, a, b, CONFIGS["completo"], id_lab=x["id"], tracer=Tracer(verbose=True))
    print({k: v for k, v in r["contadores"].items() if v}, "| tiempo", r["tiempo_s"], "s")
    dibujar(lab, r["ruta"], a, b)

## 5. Evaluación completa: 10 métodos × 20 laberintos
`aleatorio`, `baseline`, `d2_agente` (referencias del D2), `completo`, 5 ablaciones y `arbitro_regla`.
Para **reutilizar** el baseline y el agente del D2 en vez de correrlos: ejecutar antes `importar_d2("ruta/resultados.csv", OUT_DIR + "/results")`.

In [ ]:
from src.evaluate import evaluar, importar_d2, ORDEN
csv_path = evaluar(llm, ORDEN, 20, out_dir=OUT_DIR + "/results", traces_dir=OUT_DIR + "/traces", modelo=MODELO)

## 6. Resultados

In [ ]:
import pandas as pd
from src.analisis import cargar, resumen, taxonomia, critico, caso_fallo
pd.set_option("display.width", 220)
df = cargar(csv_path)
r = resumen(df)
r.to_csv(OUT_DIR + "/results/resumen.csv", index=False)
print(f"N = {df.id.nunique()} laberintos | modelo {MODELO}\n")
r.round(3)

In [ ]:
print(taxonomia(df))
c = critico(df)
c.round(3) if not c.empty else None

## 7. Caso de fallo (para el video y el PDF)
Guarda la traza del primer laberinto donde `completo` no llega a la meta.

In [ ]:
from src.maze import analizar_fallo
id_fallo = caso_fallo(df, traces_dir=OUT_DIR + "/traces", metodo="completo")
if id_fallo is not None:
    x = datos[id_fallo]
    lab, a, b = x["laberinto"], tuple(x["inicio"]), tuple(x["meta"])
    import json
    ruta = [tuple(p) for p in json.loads(df[(df.metodo == "completo") & (df.id == id_fallo)].ruta.iloc[0])]
    dibujar(lab, ruta, a, b)
    analizar_fallo(lab, ruta)

## 8. Demo para el video (≤ 3 min)
Un laberinto **nuevo**, baseline vs sistema, mensajes en vivo. Grabar esta celda completa, sin cortes.

In [ ]:
from src.demo import demo
r = demo(llm, semilla=None, config="completo", traces_dir=OUT_DIR + "/traces")

## 9. Llevar al repo
Copiar `results/` y `traces/` desde Drive a la carpeta del repo y hacer commit/push (`resultados_d3.csv`, `resumen.csv`, `traces/*.jsonl`).